# Indic Sentiment Model Training

This notebook trains and evaluates the local models of the Indic Review & Sentiment Analyzer (SMAI T8):

| # | Model | Syllabus link |
|---|---|---|
| 1 | TF-IDF + Logistic Regression | Unit II: logistic regression, L2 regularization, hyperparameter tuning |
| 2 | BiLSTM | Unit V: neural networks, back-propagation, gated recurrent network |
| 3 | MuRIL (all languages + Hinglish) | Unit VII: multilingual transformer, transfer learning |

The Unit III classifiers (Multinomial and Bernoulli Naive Bayes) are also trained on the same TF-IDF features for comparison (section 4b). A zero-shot Groq LLM (gpt-oss-20b) is evaluated as a reference. It is not trained.

**Platform.** Everything runs locally on a Windows laptop CPU in `backend/.venv` (Python 3.11, scikit-learn, PyTorch CPU, Hugging Face Transformers). No GPU or Colab is needed.

With `RUN_TRAINING = False` the notebook only displays the recorded results. Set it to `True` to retrain. A full run takes roughly 30–60 minutes on a CPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

RUN_TRAINING = False   # True = retrain everything
RUN_LLM = False        # True = also call the Groq API (needs GROQ_API_KEY in backend/.env)
PROJECT_FOLDER = None  # set this if the notebook is outside the project folder

cwd = Path.cwd()
candidates = [Path(PROJECT_FOLDER).expanduser()] if PROJECT_FOLDER else [
    cwd, *cwd.parents, cwd / 'indic-sentiment-analyzer',
]
project = next((p.resolve() for p in candidates
                if (p / 'backend/scripts/train_tfidf.py').is_file()), None)
backend = project / 'backend' if project else None
results = backend / 'artifacts/results' if backend else None
figures = backend / 'artifacts/figures' if backend else None

def run(module, *args, llm=False):
    """Run one backend script with the project's virtual environment."""
    if not RUN_TRAINING or (llm and not RUN_LLM):
        print('skipped:', module, *args)
        return
    if backend is None:
        raise RuntimeError('Set PROJECT_FOLDER to the folder that contains backend/.')
    venv_python = backend / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    python = str(venv_python if venv_python.exists() else Path(sys.executable))
    subprocess.run([python, '-u', '-m', module, *args], cwd=backend, check=True,
                   env={**os.environ, 'PYTHONUTF8': '1', 'PYTHONIOENCODING': 'utf-8'})

print('project:', project)

## 1. Recorded results

Macro-F1 on the held-out test split. The 95% confidence intervals use a group bootstrap: all translations of one source review are resampled together.

- **full**: 940 test reviews (5 Indic languages + Hinglish)
- **llm-subset**: 240 test reviews, 40 per language (LLM reference)

In [ ]:
import csv
import io

recorded_leaderboard = 'model_id,display,family,scope,n_test,accuracy,macro_f1,ci_low,ci_high,ece,n_params,train_time_s,latency_ms\nllm_groq_zs,LLM groq:openai/gpt-oss-20b (zero-shot),llm,llm-subset,240,0.9791666666666666,0.9790616112090175,0.9536530446138901,0.9958660670467404,0.08066666666666655,,,\nmuril_multi,MuRIL multilingual (all languages + Hinglish),transformer,full,940,0.9319148936170213,0.9319145853987569,0.9008761227833986,0.9582071049583252,0.05809806515561766,237557762.0,2284.585198163986,13.95999333326472\ntfidf_lr,TF-IDF (char+word n-grams) + Logistic Regression,classical,full,940,0.8276595744680851,0.8276564537157758,0.781826298548303,0.865484368332006,0.06060937961080395,205379.0,16.231570720672607,1.0472060000756755\ntfidf_nb,TF-IDF + Multinomial Naive Bayes,classical,full,940,0.8127659574468085,0.8127651098465816,0.7666374127942114,0.8544834236479069,0.05090720984936849,410758.0,11.943183183670044,3.6576340001192875\nbilstm_subword,"BiLSTM (subword-local tokens, maxmean pooling, 1 layer)",rnn,full,940,0.798936170212766,0.7988157237640008,0.7579710871683724,0.8300637657134148,0.1684536139381692,1292674.0,321.7538073062897,4.357182000239845\ntfidf_bnb,TF-IDF n-grams + Bernoulli Naive Bayes (presence/absence),classical,full,940,0.7595744680851064,0.7583833384134553,0.7064691852956042,0.8055943905889857,0.2386656201864854,410758.0,12.238362073898315,12.18001600005664\n'
recorded_per_language = ',bn,hi,hinglish,mr,ta,te\nllm_groq_zs,0.9746031746031746,0.9494949494949495,1.0,1.0,1.0,0.948849104859335\nmuril_multi,0.9537556802993852,0.9306335204490778,0.9196428571428572,0.9248454469507101,0.9421791443850267,0.9132832080200501\ntfidf_lr,0.8090188338406985,0.8439097744360902,0.8092296511627908,0.8258856682769726,0.8265838011226945,0.8381448810478482\ntfidf_nb,0.8437426822332987,0.7742949854480983,0.76515011972739,0.8205934499715652,0.8207852965747702,0.8208092485549133\nbilstm_subword,0.803409090909091,0.8265838011226945,0.7450348899624262,0.7794551798174987,0.8264446227929374,0.7797507370678103\ntfidf_bnb,0.7842820072119435,0.6940136054421768,0.7437511238985794,0.7370376625570005,0.7797507370678103,0.7972543110664658\n'

live = results / 'leaderboard.csv' if results else None
leaderboard = live.read_text(encoding='utf-8') if live and live.is_file() else recorded_leaderboard
live_pl = results / 'per_language_macro_f1.csv' if results else None
per_language = live_pl.read_text(encoding='utf-8') if live_pl and live_pl.is_file() else recorded_per_language

print(f"{'Model':16} {'Scope':11} {'Test n':>6} {'Macro-F1':>9} {'95% CI':>13} {'ECE':>6}")
for r in csv.DictReader(io.StringIO(leaderboard)):
    print(f"{r['model_id']:16} {r['scope']:11} {r['n_test']:>6} {float(r['macro_f1']):9.3f} "
          f"  {float(r['ci_low']):.3f}-{float(r['ci_high']):.3f} {float(r['ece']):6.3f}")

print('\nMacro-F1 per language')
rows = list(csv.reader(io.StringIO(per_language)))
print(f"{'Model':16}" + ''.join(f'{h:>10}' for h in rows[0][1:]))
for r in rows[1:]:
    print(f'{r[0]:16}' + ''.join(f'{float(v):10.3f}' if v else f"{'-':>10}" for v in r[1:]))

## 2. Prepare the data (Unit I: supervised learning)

IndicSentiment provides parallel reviews: the same English review translated into each language. All translations of one review are assigned to the same split (70% train / 15% validation / 15% test). Otherwise the model could see a Hindi translation in training and its Bengali translation in testing. The script checks that no review group crosses splits.

In [ ]:
run('scripts.download_data')
run('scripts.prepare_data')

## 3. Generate Hinglish data (T8.5, uses the LLM)

Five hundred English source reviews are rewritten as Roman-script Hinglish and labelled by a second LLM call. Train and validation rows use the LLM label (weak supervision). Test rows keep the gold label. The recorded run reached 99% agreement with the gold labels (Cohen's kappa 0.98). Skip this step if `backend/data/processed/hinglish.csv` already exists.

In [ ]:
run('scripts.make_hinglish', '--n', '500', llm=True)

## 4. Model 1: TF-IDF + Logistic Regression (Unit II)

Text becomes a sparse vector of character 2–5-grams and word 1–2-grams, weighted by TF-IDF. Logistic regression minimizes cross-entropy loss with L2 regularization. The regularization strength `C` is chosen from {0.25, 0.5, 1, 2, 4, 8} by validation macro-F1.

In [ ]:
run('scripts.train_tfidf')

## 4b. Syllabus classifiers on the same TF-IDF features (Unit III)

The same TF-IDF features are fed to the classifiers from Unit III, so only the classifier changes. Each hyperparameter is chosen by validation macro-F1.

| Classifier | Idea | Tuned |
|---|---|---|
| Multinomial Naive Bayes | Bayes' rule with independent features; per-class multinomial over n-gram weights | smoothing `alpha` |
| Bernoulli Naive Bayes | the same rule, but each n-gram is only present or absent, and absent n-grams also count as evidence | smoothing `alpha` |

In [ ]:
for clf in ['nb', 'bnb']:
    run('scripts.train_tfidf', '--clf', clf)

## 5. Model 2: BiLSTM (Unit V)

A SentencePiece tokenizer (8k subwords) is fitted on the training reviews. The model has these layers: embedding, then a bidirectional LSTM layer (128 hidden units), then masked mean + max pooling, then dropout, then a linear softmax layer.

- **Training:** back-propagation through time with AdamW (learning rate 2e-3, weight decay 1e-4), class-weighted cross-entropy and gradient clipping.
- **Stopping:** up to 20 epochs, stopping early after 4 epochs without improvement in validation macro-F1.
- **The cell:** each LSTM step uses input, forget and output gates to decide what to remember, and the two directions read the review left-to-right and right-to-left.

In [ ]:
run('scripts.train_rnn', '--cell', 'lstm', '--bidirectional', '--model-id', 'bilstm_subword')

## 6. Model 3: MuRIL, one transformer for all languages (Unit VII)

`google/muril-base-cased` is a BERT model pretrained on 17 Indian languages, including transliterated (Roman-script) text, so it also covers Hinglish. It is fine-tuned on all 4,380 training reviews with these settings: 4 epochs, learning rate 3e-5, linear warm-up, 128 tokens and a frozen input embedding table. This is the single transformer that serves every language in the app.

In [ ]:
run('scripts.train_transformer', '--model-name', 'google/muril-base-cased', '--model-id', 'muril_multi',
    '--display', 'MuRIL multilingual (all languages + Hinglish)',
    '--epochs', '4', '--max-len', '128', '--freeze-embeddings')

## 7. LLM reference (zero-shot and 3-shot, aspects)

These steps call the Groq API and use about 200k tokens in total, which is the free tier's daily limit. Responses are cached in `backend/artifacts/llm_cache`, so a rerun only makes the calls that are missing.

In [ ]:
run('scripts.eval_llm', '--providers', 'groq', '--shots', '0', llm=True)
run('scripts.eval_llm', '--providers', 'groq', '--shots', '3', llm=True)
run('scripts.eval_aspects', '--n', '30', '--provider', 'groq', llm=True)

## 8. Evaluate (Unit IV)

This step rebuilds the leaderboard, the per-language table, the McNemar significance tests, the expected-versus-actual check and all figures.

In [ ]:
run('scripts.evaluate_all')
run('scripts.compare_expected')

## 9. Figures

In [ ]:
from IPython.display import Image, display

for name in ['fig_macro_f1.png', 'fig_per_language_heatmap.png', 'fig_confusion_matrices.png',
             'fig_ablations.png', 'fig_training_curves.png', 'fig_reliability.png']:
    path = figures / name if figures else None
    if path and path.is_file():
        print(name)
        display(Image(filename=str(path)))